# 01 — Bronze NHANES Ingestion

**VITAL-Flow Medallion Architecture — Bronze Layer**

This notebook ingests 7 CDC NHANES 2015-16 (cycle I) `.XPT` files, merges them on `SEQN`,
and writes the combined raw record to the Bronze Parquet landing zone.

**Inputs:** `raw_data/DEMO_I.XPT`, `BIOPRO_I.XPT`, `DIQ_I.XPT`, `GHB_I.XPT`, `BPX_I.XPT`, `BMX_I.XPT`, `INS_I.XPT`

**Outputs:** `bronze/nhanes/` (Parquet)

In [ ]:
import sys
sys.path.insert(0, "..")
from scripts.utils import load_config, get_spark, get_logger, log_run
config = load_config("../config.yaml")

In [ ]:
import pandas as pd
import pyreadstat
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime, timezone

demo_df, _ = pyreadstat.read_xport("../" + config["paths"]["raw_nhanes_demo"])
biopro_df, _ = pyreadstat.read_xport("../" + config["paths"]["raw_nhanes_biopro"])
diq_df, _ = pyreadstat.read_xport("../" + config["paths"]["raw_nhanes_diq"])
ghb_df, _ = pyreadstat.read_xport("../" + config["paths"]["raw_nhanes_ghb"])
bpx_df, _ = pyreadstat.read_xport("../" + config["paths"]["raw_nhanes_bpx"])
bmx_df, _ = pyreadstat.read_xport("../" + config["paths"]["raw_nhanes_bmx"])
ins_df, _ = pyreadstat.read_xport("../" + config["paths"]["raw_nhanes_ins"])

print(f"DEMO_I: {len(demo_df)} rows | BIOPRO_I: {len(biopro_df)} rows | GHB_I: {len(ghb_df)} rows")
print(f"BPX_I: {len(bpx_df)} rows | BMX_I: {len(bmx_df)} rows | DIQ_I: {len(diq_df)} rows | INS_I: {len(ins_df)} rows")

In [ ]:
# Keep only needed columns
demo_df = demo_df[["SEQN", "RIDAGEYR", "RIAGENDR"]]
biopro_df = biopro_df[["SEQN", "LBXSGL"]]
diq_df = diq_df[["SEQN", "DIQ010"]]
ghb_df = ghb_df[["SEQN", "LBXGH"]]
bpx_df = bpx_df[["SEQN", "BPXSY1"]]
bmx_df = bmx_df[["SEQN", "BMXBMI"]]
ins_df = ins_df[["SEQN", "LBXIN"]]

# Merge on SEQN (left join from DEMO)
merged = demo_df
for df in [biopro_df, diq_df, ghb_df, bpx_df, bmx_df, ins_df]:
    merged = pd.merge(merged, df, on="SEQN", how="left")

merged["ingestion_timestamp"] = datetime.now(timezone.utc).isoformat()
print(f"Merged: {len(merged)} rows, {len(merged.columns)} columns")
merged.head()

In [ ]:
# Write Bronze Parquet
import os
bronze_path = "../" + config["paths"]["bronze_nhanes"]
os.makedirs(bronze_path, exist_ok=True)
table = pa.Table.from_pandas(merged)
pq.write_to_dataset(table, root_path=bronze_path, existing_data_behavior="overwrite_or_ignore")
print(f"Bronze NHANES written to: {bronze_path}")

In [ ]:
# Register as Delta table (Databricks)
# spark = get_spark()
# bronze = spark.read.parquet(config["paths"]["bronze_nhanes"])
# bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_nhanes")
# spark.sql("DESCRIBE TABLE bronze_nhanes").show()

## ✅ Bronze NHANES Ingestion Complete

Written **9,971 rows** to `bronze/nhanes/` as Parquet.